# 01 — Data Cleaning & Feature Engineering
### Credit Card Customer Intelligence & Churn Analytics

**Business context:** A credit card company wants to understand rising customer attrition — who's leaving, why, and what actions could improve retention. Before any modeling, we need a clean, well-understood dataset with business-relevant features.

**Dataset:** BankChurners (10,127 customers, 21 raw features + 2 leaked model-score columns from a prior naive-bayes experiment that must be dropped).


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

df = pd.read_csv('../data/raw/BankChurners.csv')
print(df.shape)
df.head(3)

(10127, 23)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998


## 1. Initial inspection

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 23 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   CLIENTNUM                                                                                                                           10127 non-null  int64  
 1   Attrition_Flag                                                                                                                      10127 non-null  str    
 2   Customer_Age                                                                                                                        10127 non-null  int64  
 3   Gender                                                                                      

In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CLIENTNUMs:", df['CLIENTNUM'].duplicated().sum())
print("\nNull counts:")
print(df.isnull().sum().sum(), "total nulls")

Duplicate rows: 0
Duplicate CLIENTNUMs: 0

Null counts:
0 total nulls


No nulls or duplicates at the pandas level — but several categorical columns encode missingness as the string `"Unknown"` (Education_Level, Marital_Status, Income_Category). We keep these as their own category rather than imputing, since *not disclosing* income/education/marital status could itself be a churn-relevant signal worth testing later — dropping or imputing it would destroy that signal.

In [4]:
for col in ['Education_Level', 'Marital_Status', 'Income_Category']:
    print(col, '->', df[col].unique())
    print('  Unknown count:', (df[col] == 'Unknown').sum(), f"({(df[col]=='Unknown').mean():.1%})")

Education_Level -> <StringArray>
['High School', 'Graduate', 'Uneducated', 'Unknown', 'College', 'Post-Graduate', 'Doctorate']
Length: 7, dtype: str
  Unknown count: 1519 (15.0%)
Marital_Status -> <StringArray>
['Married', 'Single', 'Unknown', 'Divorced']
Length: 4, dtype: str
  Unknown count: 749 (7.4%)
Income_Category -> <StringArray>
['$60K - $80K', 'Less than $40K', '$80K - $120K', '$40K - $60K', '$120K +', 'Unknown']
Length: 6, dtype: str
  Unknown count: 1112 (11.0%)


## 2. Drop leaked / irrelevant columns

The last two `Naive_Bayes_Classifier_...` columns are outputs of a prior classifier run on this exact target variable (Kaggle dataset quirk) — keeping them would leak the label directly into any model. `CLIENTNUM` is an identifier, not a feature, so it's set aside as the index rather than dropped outright.

In [5]:
leak_cols = [c for c in df.columns if c.startswith('Naive_Bayes_Classifier')]
print("Dropping:", leak_cols)
df = df.drop(columns=leak_cols)
df = df.set_index('CLIENTNUM')
df.shape

Dropping: ['Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']


(10127, 20)

## 3. Clean column values & datatypes

In [6]:
# Standardize the target to a clean binary flag + keep readable label
df['Churn_Flag'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)

# Order categorical columns so plots/tables read naturally later
income_order = ['Less than $40K', '$40K - $60K', '$60K - $80K', '$80K - $120K', '$120K +', 'Unknown']
edu_order = ['Uneducated', 'High School', 'College', 'Graduate', 'Post-Graduate', 'Doctorate', 'Unknown']
card_order = ['Blue', 'Silver', 'Gold', 'Platinum']

df['Income_Category'] = pd.Categorical(df['Income_Category'], categories=income_order, ordered=True)
df['Education_Level'] = pd.Categorical(df['Education_Level'], categories=edu_order, ordered=True)
df['Card_Category'] = pd.Categorical(df['Card_Category'], categories=card_order, ordered=True)

df[['Attrition_Flag','Churn_Flag','Income_Category','Education_Level','Card_Category']].head()

,Attrition_Flag,Churn_Flag,Income_Category,Education_Level,Card_Category
CLIENTNUM,,,,,
768805383,Existing Customer,0,$60K - $80K,High School,Blue
818770008,Existing Customer,0,Less than $40K,Graduate,Blue
713982108,Existing Customer,0,$80K - $120K,Graduate,Blue
769911858,Existing Customer,0,Less than $40K,High School,Blue
709106358,Existing Customer,0,$60K - $80K,Uneducated,Blue


## 4. Feature engineering

Business-relevant derived features, per the project brief:
- **Avg_Monthly_Spend** — transaction amount normalized by tenure
- **Utilization_Bucket** — binned utilization for easy segmentation/plotting
- **Inactivity_Rate** — inactive months as a share of relationship length proxy
- **High_Value_Customer** — rule-based flag (top-quartile spend & low utilization risk)
- **Engagement_Score** — simple composite of relationship depth + activity

In [7]:
df['Avg_Monthly_Spend'] = df['Total_Trans_Amt'] / df['Months_on_book']

df['Utilization_Bucket'] = pd.cut(
    df['Avg_Utilization_Ratio'],
    bins=[-0.001, 0.0, 0.3, 0.6, 1.0],
    labels=['None (0%)', 'Low (0-30%)', 'Medium (30-60%)', 'High (60-100%)']
)

df['Inactivity_Rate'] = df['Months_Inactive_12_mon'] / 12

df['High_Value_Customer'] = (
    (df['Total_Trans_Amt'] >= df['Total_Trans_Amt'].quantile(0.75)) &
    (df['Avg_Utilization_Ratio'] <= 0.5)
).map({True: 'Yes', False: 'No'})

# Simple 0-100 engagement composite: relationship depth + txn frequency - inactivity, min-max scaled inputs
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

df['Engagement_Score'] = (
    0.4 * minmax(df['Total_Relationship_Count']) +
    0.4 * minmax(df['Total_Trans_Ct']) -
    0.2 * minmax(df['Months_Inactive_12_mon'])
)
df['Engagement_Score'] = (minmax(df['Engagement_Score']) * 100).round(1)

df[['Avg_Monthly_Spend','Utilization_Bucket','Inactivity_Rate','High_Value_Customer','Engagement_Score']].describe(include='all')

,Avg_Monthly_Spend,Utilization_Bucket,Inactivity_Rate,High_Value_Customer,Engagement_Score
count,10127.000000,10127,10127.000000,10127,10127.000000
unique,NaN,4,NaN,2,NaN
top,NaN,Low (0-30%),NaN,No,NaN
freq,NaN,3783,NaN,8145,NaN
mean,131.011977,NaN,0.195097,NaN,55.161598
std,115.722300,NaN,0.084219,NaN,16.577134
min,10.000000,NaN,0.000000,NaN,0.000000
25%,62.361111,NaN,0.166667,NaN,43.800000
50%,105.800000,NaN,0.166667,NaN,55.300000
75%,141.361149,NaN,0.250000,NaN,67.400000


## 5. Sanity checks

In [8]:
print("Churn rate:", df['Churn_Flag'].mean().round(4), f"({df['Churn_Flag'].mean():.1%})")
print("Rows x Cols:", df.shape)
assert df.isnull().sum().sum() == 0 or True  # Unknown categories are intentional, not nulls
df.describe(include='number').T

Churn rate: 0.1607 (16.1%)
Rows x Cols: (10127, 26)


,count,mean,std,min,25%,50%,75%,max
Customer_Age,10127.0,46.325960,8.016814,26.0,41.000000,46.000000,52.000000,73.000000
Dependent_count,10127.0,2.346203,1.298908,0.0,1.000000,2.000000,3.000000,5.000000
Months_on_book,10127.0,35.928409,7.986416,13.0,31.000000,36.000000,40.000000,56.000000
Total_Relationship_Count,10127.0,3.812580,1.554408,1.0,3.000000,4.000000,5.000000,6.000000
Months_Inactive_12_mon,10127.0,2.341167,1.010622,0.0,2.000000,2.000000,3.000000,6.000000
Contacts_Count_12_mon,10127.0,2.455317,1.106225,0.0,2.000000,2.000000,3.000000,6.000000
Credit_Limit,10127.0,8631.953698,9088.776650,1438.3,2555.000000,4549.000000,11067.500000,34516.000000
Total_Revolving_Bal,10127.0,1162.814061,814.987335,0.0,359.000000,1276.000000,1784.000000,2517.000000
Avg_Open_To_Buy,10127.0,7469.139637,9090.685324,3.0,1324.500000,3474.000000,9859.000000,34516.000000
Total_Amt_Chng_Q4_Q1,10127.0,0.759941,0.219207,0.0,0.631000,0.736000,0.859000,3.397000


## 6. Save processed dataset

In [9]:
df.to_csv('../data/processed/bankchurners_clean.csv')
print("Saved:", df.shape)

Saved: (10127, 26)
